# Local DataMorgana-Style Synthetic QA Generation

This notebook replicates the DataMorgana synthetic QA generation pipeline
([Filice et al., 2025](https://arxiv.org/abs/2501.12789)) **without requiring
access to the ai71 API or the DataMorgana service**.

Instead, it uses a locally-downloaded HuggingFace model (e.g. Llama-3,
Mistral, Falcon-3) and your own corpus of documents.

## Pipeline summary
1. Define **question and user categorizations** (same schema as DataMorgana)
2. Point to a **local corpus** directory of `.txt` or `.jsonl` files
3. Load a **local LLM** via HuggingFace `transformers`
4. Run generation → each QA pair is labelled with its sampled category combination
5. Filter candidates for basic quality / faithfulness
6. Save results as JSONL (compatible with downstream RAG evaluation scripts)

---

## 0. Imports & setup

In [ ]:
import json, random, sys, logging
from pathlib import Path

# Make sure datamorgana_local.py is importable from this notebook
sys.path.insert(0, str(Path('.').resolve()))

from datamorgana_local import (
    sample_categories,
    build_prompt,
    filter_candidates,
    load_corpus,
    LocalLLM,
    generate_synthetic_qa,
)

logging.basicConfig(format='%(asctime)s | %(levelname)s | %(message)s', level=logging.INFO)
random.seed(42)
print('Imports OK')

---
## 1. Define categorizations

You can define any number of question and user categorizations.  
The key rules (from the DataMorgana paper):

- Categories within a single categorization must be **mutually exclusive**.
- Probabilities within a categorization must **sum to 1.0**.
- The **`description`** field is injected verbatim into the LLM prompt — write it carefully.

The cells below show the exact categorizations used in the CIKM paper.

In [ ]:
# ── QUESTION CATEGORIZATIONS ────────────────────────────────────────────────

question_factuality = {
    'categorization_name': 'question_factuality',
    'categories': [
        {'name': 'factoid',
         'description': 'A question seeking a specific, concise piece of information or a short fact about a particular subject, such as a name, date, or number.',
         'probability': 0.5},
        {'name': 'open_ended',
         'description': 'A question inviting detailed or exploratory responses, encouraging discussion or elaboration.',
         'probability': 0.5},
    ]
}

question_premise = {
    'categorization_name': 'question_premise',
    'categories': [
        {'name': 'direct',
         'description': 'A question that does not contain any premise or any information about the user.',
         'probability': 0.5},
        {'name': 'with_premise',
         'description': 'A question starting with a very short premise, where the user reveals their needs or some information about themselves.',
         'probability': 0.5},
    ]
}

question_phrasing = {
    'categorization_name': 'question_phrasing',
    'categories': [
        {'name': 'concise_and_natural',
         'description': 'A question phrased in the way people typically speak, reflecting everyday language use, without formal or artificial structure. It is a concise direct question consisting of less than 10 words.',
         'probability': 0.25},
        {'name': 'verbose_and_natural',
         'description': 'A question phrased in the way people typically speak, reflecting everyday language use, without formal or artificial structure. It is a relatively long question consisting of more than 9 words.',
         'probability': 0.25},
        {'name': 'short_search_query',
         'description': 'A question phrased as a typed web query for search engines (only keywords, without punctuation and without a natural-sounding structure). It consists of less than 7 words.',
         'probability': 0.25},
        {'name': 'long_search_query',
         'description': 'A question phrased as a typed web query for search engines (only keywords, without punctuation and without a natural-sounding structure). It consists of more than 6 words.',
         'probability': 0.25},
    ]
}

question_linguisticvariation = {
    'categorization_name': 'question_linguisticvariation',
    'categories': [
        {'name': 'similar_to_document',
         'description': 'A question phrased using the same terminology and phrases appearing in the document.',
         'probability': 0.5},
        {'name': 'distant_from_document',
         'description': 'A question phrased using terms completely different from the ones appearing in the document.',
         'probability': 0.5},
    ]
}

# ── USER CATEGORIZATIONS ────────────────────────────────────────────────────

user_expertise = {
    'categorization_name': 'user_expertise',
    'categories': [
        {'name': 'expert',
         'description': 'A specialized user with deep understanding of the corpus.',
         'probability': 0.5},
        {'name': 'novice',
         'description': 'A regular user with no understanding of specialized terms.',
         'probability': 0.5},
    ]
}

# ── BUNDLE ──────────────────────────────────────────────────────────────────
# Swap / add / remove categorizations here to match your experimental design

QUESTION_CATS = [
    question_factuality,
    question_premise,
    question_phrasing,
    question_linguisticvariation,
]

USER_CATS = [user_expertise]

print(f'Defined {len(QUESTION_CATS)} question categorizations, {len(USER_CATS)} user categorizations.')

---
## 2. Load your local corpus

The corpus loader accepts:
- **`.txt` files** — one document per file
- **`.jsonl` files** — one JSON object per line, must have a `text` (or `content` / `passage`) field

To replicate the FineWeb-10BT experiments, download a portion of the dataset from HuggingFace
(`HuggingFaceFW/fineweb`, `sample-10BT` split, ~27 GB) and convert to `.jsonl`.  
For a quick test, the `sample_corpus/` directory contains three example documents.

In [ ]:
CORPUS_DIR = './sample_corpus'   # ← change to your corpus directory

corpus = load_corpus(CORPUS_DIR)
print(f'Loaded {len(corpus)} documents.')
print('\nFirst document preview:')
print(corpus[0]['text'][:400] + ' …')

---
## 3. Inspect the prompt template

The cell below samples one category combination and builds the full prompt for a randomly chosen document.  
This is useful for verifying that your category descriptions are clear and well-formed **before** running the model.

In [ ]:
q_selected, q_combo = sample_categories(QUESTION_CATS)
u_selected, u_combo = sample_categories(USER_CATS)

doc = random.choice(corpus)
prompt = build_prompt(
    document=doc['text'],
    question_categories=q_selected,
    user_categories=u_selected,
    k=3,
)

print('=== SAMPLED COMBINATION ===')
print(f'User:     {u_combo}')
print(f'Question: {q_combo}')
print()
print('=== PROMPT SENT TO LLM ===')
print(prompt)

---
## 4. Load the local LLM

Any instruction-tuned HuggingFace model works. Recommended options:

| Model | VRAM | Notes |
|---|---|---|
| `meta-llama/Meta-Llama-3.1-8B-Instruct` | ~16 GB | Best quality/size trade-off |
| `mistralai/Mistral-7B-Instruct-v0.3` | ~14 GB | Fast, reliable |
| `tiiuae/Falcon3-10B-Instruct` | ~20 GB | Used in original CIKM paper |
| `Qwen/Qwen2.5-7B-Instruct` | ~14 GB | Strong multilingual support |

Add `load_in_4bit=True` if you have limited VRAM (requires `bitsandbytes`).

In [ ]:
MODEL = 'meta-llama/Meta-Llama-3.1-8B-Instruct'   # ← change to your preferred model

llm = LocalLLM(
    model_name_or_path=MODEL,
    device='auto',           # uses GPU(s) if available
    max_new_tokens=1024,
    temperature=0.7,
    top_p=0.9,
    load_in_4bit=False,      # set True for 4-bit quantisation
)
print('Model loaded.')

---
## 5. Dry-run: generate one QA pair manually

Before running the full batch it is always worth generating a single pair
to check that the model output parses correctly and looks reasonable.

In [ ]:
doc = random.choice(corpus)
q_sel, _ = sample_categories(QUESTION_CATS)
u_sel, _ = sample_categories(USER_CATS)

prompt = build_prompt(doc['text'], q_sel, u_sel, k=3)
raw_output = llm.generate(prompt)

print('=== RAW MODEL OUTPUT ===')
print(raw_output)
print()

candidates = filter_candidates(raw_output, doc['text'])
print(f'=== FILTERED CANDIDATES ({len(candidates)}) ===')
for i, c in enumerate(candidates, 1):
    print(f'\n--- Candidate {i} ---')
    print(f'Q: {c["question"]}')
    print(f'A: {c["answer"]}')

---
## 6. Full batch generation

Run this cell to generate `N_QUESTIONS` synthetic QA pairs.
Results are saved to `OUTPUT_FILE` in JSONL format.

In [ ]:
N_QUESTIONS  = 200       # ← total number of QA pairs to generate
K_CANDIDATES = 3         # ← candidate pairs per document (then filter to 1)
OUTPUT_FILE  = './synthetic_qa_output.jsonl'

results = generate_synthetic_qa(
    corpus=corpus,
    question_categorizations=QUESTION_CATS,
    user_categorizations=USER_CATS,
    llm=llm,
    n_questions=N_QUESTIONS,
    k_candidates=K_CANDIDATES,
)

with open(OUTPUT_FILE, 'w', encoding='utf-8') as fh:
    for item in results:
        fh.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f'\n✓ Saved {len(results)} QA pairs to {OUTPUT_FILE}')

---
## 7. Inspect outputs

In [ ]:
# Load and display a random sample of generated QA pairs
import random

with open(OUTPUT_FILE) as fh:
    all_results = [json.loads(l) for l in fh]

sample = random.sample(all_results, min(5, len(all_results)))

for item in sample:
    print('─' * 70)
    print(f"Q  : {item['question']}")
    print(f"A  : {item['answer']}")
    print(f"DOC: {item['document_id']}")
    print(f"CAT: {item['combo_key']}")

In [ ]:
# Distribution of category combinations actually generated
from collections import Counter

combo_counts = Counter(item['combo_key'] for item in all_results)
print(f'Total unique combinations: {len(combo_counts)}')
print(f'Total questions generated: {len(all_results)}\n')

print('Top-10 most frequent combinations:')
for combo, count in combo_counts.most_common(10):
    print(f'  {count:4d}  {combo}')

---
## 8. Custom categorizations — quick-start examples

The following cells show two alternative configurations you can swap in above.

In [ ]:
# ── Example A: Question Complexity hierarchy (from RQ1 in the extended paper) ──

question_complexity_medium = {
    'categorization_name': 'question_complexity',
    'categories': [
        {'name': 'single_fact',
         'description': 'A question asking for one specific fact, date, name, or piece of information that appears explicitly in one location in the document.',
         'probability': 0.25},
        {'name': 'multi_fact_local',
         'description': 'A question requiring multiple facts or pieces of information that appear within the same paragraph or closely related section of the document.',
         'probability': 0.25},
        {'name': 'cross_section_synthesis',
         'description': 'A question requiring the reader to connect or synthesize information from multiple distinct sections of the document that are not immediately adjacent.',
         'probability': 0.25},
        {'name': 'comparative_analysis',
         'description': 'A question requiring comparison, contrast, or relational analysis between different entities, concepts, time periods, or perspectives discussed in the document.',
         'probability': 0.25},
    ]
}

# Use: QUESTION_CATS = [question_complexity_medium]

In [ ]:
# ── Example B: Domain-specific user roles for a healthcare corpus ──

user_healthcare = {
    'categorization_name': 'user_role',
    'categories': [
        {'name': 'patient',
         'description': 'A regular patient who uses the system to get basic health information and understand treatment options in accessible language.',
         'probability': 0.3},
        {'name': 'medical_doctor',
         'description': 'A medical doctor who needs to access advanced clinical information and treatment protocols.',
         'probability': 0.35},
        {'name': 'clinical_researcher',
         'description': 'A clinical researcher who uses the system to access population health data and research findings.',
         'probability': 0.2},
        {'name': 'public_health_authority',
         'description': 'A public health authority who uses the system to manage community health information and respond to health emergencies.',
         'probability': 0.15},
    ]
}

# Use: USER_CATS = [user_healthcare]

---
## References

- Filice, S., Horowitz, G., Carmel, D., Karnin, Z., Lewin-Eytan, L., & Maarek, Y. (2025). *Generating Diverse Q&A Benchmarks for RAG Evaluation with DataMorgana*. arXiv:2501.12789.
- Fensore, C., Dhole, K., Agichtein, E., & Ho, J. (2025). *Investigating Hybrid Retrieval Strategies for Augmented Generation over Diverse Dynamic Test Sets*. CIKM 2025.